# Práctica 03 — Distribuciones empíricas

**Curso:** Estadística Aplicada con Python · **Clase 03** · **Tiempo estimado:** 60-90 minutos
**Notebook de clase:** `notebooks_clase/clase03_distribuciones_empiricas.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te dice qué revisar. Si ves "Completa las líneas marcadas con TODO", todavía te falta rellenar algo.
- Las **soluciones** están al final, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden: algunos ejercicios reutilizan variables de los anteriores.
- Datos: `penguins` (*Palmer Penguins*, Horst, Hill y Gorman). Si no hay copia local ni conexión se usa un dataset sintético (*ejemplo ilustrativo*): las comprobaciones funcionan igual con ambos.

---
## Celda 0 — Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
sns.set_theme(style="whitegrid")
print("Librerías cargadas correctamente.")

In [ ]:
def generar_penguins_sintetico(semilla=2026):
    """Genera un dataset SINTÉTICO con las mismas columnas que `penguins`.

    EJEMPLO ILUSTRATIVO: los valores imitan el orden de magnitud de los datos reales,
    pero NO son pingüinos reales. Solo sirve como alternativa sin conexión."""
    r = np.random.default_rng(semilla)
    # por especie: n, (media, desviación) de cada medida y las islas posibles
    especies = {
        "Adelie":    dict(n=152, pico=(38.8, 2.7), prof=(18.3, 1.2), aleta=(190, 6.5), masa=(3700, 460), islas=["Torgersen", "Biscoe", "Dream"]),
        "Chinstrap": dict(n=68,  pico=(48.8, 3.3), prof=(18.4, 1.1), aleta=(196, 7.1), masa=(3733, 384), islas=["Dream"]),
        "Gentoo":    dict(n=124, pico=(47.5, 3.1), prof=(15.0, 1.0), aleta=(217, 6.5), masa=(5076, 504), islas=["Biscoe"]),
    }
    bloques = []
    for esp, c in especies.items():
        n = c["n"]
        sexo = r.choice(["Male", "Female"], size=n)
        efecto = np.where(sexo == "Male", 0.5, -0.5)          # los machos son algo mayores que las hembras
        def medida(par):
            return par[0] + efecto * par[1] + r.normal(0, par[1] * 0.8, size=n)
        bloques.append(pd.DataFrame({
            "species": esp,
            "island": r.choice(c["islas"], size=n),
            "bill_length_mm": np.round(medida(c["pico"]), 1),
            "bill_depth_mm": np.round(medida(c["prof"]), 1),
            "flipper_length_mm": np.round(medida(c["aleta"])),
            "body_mass_g": np.round(medida(c["masa"]) / 25) * 25,     # la masa real viene redondeada a 25 g
            "sex": sexo,
        }))
    d = pd.concat(bloques, ignore_index=True)
    idx = r.choice(len(d), size=11, replace=False)               # filas 0 y 1 sin ninguna medida; 9 filas más sin sexo (como en los datos reales)
    d.loc[idx[:2], ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex"]] = np.nan
    d.loc[idx[2:], "sex"] = np.nan
    return d


def cargar_penguins():
    """Carga penguins: copia local -> descarga de seaborn -> sintético."""
    for ruta in ("datos/penguins.csv", "../datos/penguins.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "copia local (" + ruta + ")"
    try:
        return sns.load_dataset("penguins"), "descarga desde seaborn"
    except Exception:
        return generar_penguins_sintetico(), "SINTÉTICO (ejemplo ilustrativo, no son pingüinos reales)"


def completos(df, columnas):
    """Devuelve las filas sin ausentes en `columnas` e INFORMA de cuántas se excluyen."""
    sub = df.dropna(subset=columnas)
    cols = ", ".join(columnas)
    print(f"[{cols}] filas usadas: {len(sub)} de {len(df)}  (excluidas por ausentes: {len(df) - len(sub)})")
    return sub

## Ejercicio 1 — Valores ausentes: cuéntalos, no los escondas

Con el dataset `penguins` (ya cargado en `df`):

- `faltantes`: Serie con el número de ausentes de **cada columna**,
- `n_filas_completas`: cuántas filas no tienen ningún ausente,
- `col_mas_ausentes`: el nombre de la columna con más ausentes,
- `aleta`: la longitud de aleta **sin ausentes** (una Serie; usa la función `completos`).

In [ ]:
df, origen = cargar_penguins()
print("Origen de los datos:", origen)

faltantes = None            # TODO: Serie con el número de ausentes de cada columna
n_filas_completas = None    # TODO: cuántas filas NO tienen ningún ausente
col_mas_ausentes = None     # TODO: nombre de la columna con más ausentes
aleta = None                # TODO: longitud de aleta sin ausentes (Serie), sin tocar el resto de columnas

# Comprobación automática
assert None not in (n_filas_completas, col_mas_ausentes) and faltantes is not None and aleta is not None, "Completa las cuatro líneas marcadas con TODO"
assert int(faltantes.sum()) == int(df.isna().to_numpy().sum()), "faltantes debe contar los ausentes de cada columna"
assert n_filas_completas == int(df.notna().all(axis=1).sum()), "n_filas_completas: filas sin ningún ausente"
assert faltantes[col_mas_ausentes] == faltantes.max(), "col_mas_ausentes debe ser la columna con más ausentes"
assert aleta.notna().all() and len(aleta) == int(df["flipper_length_mm"].notna().sum()), "aleta debe ser la columna de aleta sin ausentes"
print("Columna con más ausentes:", col_mas_ausentes, "|", "Filas completas:", n_filas_completas)
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Un histograma a mano

Con los doce datos del capítulo, calcula los conteos con intervalos de ancho 5 (bordes 0, 5, 10, 15, 20, 25) y con ancho 10 (bordes 0, 10, 20, 30), y las **proporciones** del primer caso.

In [ ]:
x = np.array([2, 3, 7, 8, 8, 9, 12, 13, 13, 14, 21, 22])

bordes = None        # TODO: bordes 0, 5, 10, 15, 20, 25 (pista: np.arange)
conteos = None       # TODO: conteos de cada intervalo (pista: np.histogram)
proporciones = None  # TODO: cada conteo dividido entre el total de datos
conteos10 = None     # TODO: conteos con intervalos de ancho 10 (bordes 0, 10, 20, 30)

# Comprobación automática
assert None not in (bordes, conteos, proporciones, conteos10), "Completa las cuatro líneas marcadas con TODO"
assert list(conteos) == [2, 4, 4, 0, 2], "los conteos con ancho 5 deben ser [2, 4, 4, 0, 2]"
assert np.isclose(np.sum(proporciones), 1.0), "las proporciones deben sumar 1"
assert list(conteos10) == [6, 4, 2], "con ancho 10 los conteos deben ser [6, 4, 2]"
print("Con ancho 5 se ve el hueco [15,20); con ancho 10 desaparece.")
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — ECDF a mano

Completa la función `ecdf(valores)`, que devuelve los valores **ordenados** y la proporción acumulada (1/n, 2/n, …, n/n). Después usa los mismos doce datos para calcular `p_13` (proporción de datos ≤ 13) y `mediana_ecdf` (primer valor donde la ECDF llega a 0,5 o más). Por último, con la masa corporal de `penguins`, calcula la proporción de pingüinos con masa ≤ 4 000 g.

In [ ]:
def ecdf(valores):
    """Devuelve (x ordenados, proporción acumulada) para cada dato."""
    xs = None    # TODO: ordena los valores
    ys = None    # TODO: 1/n, 2/n, ..., n/n
    return xs, ys

xs, ys = ecdf(x)
assert xs is not None and ys is not None, "Completa la función (TODO)"

p_13 = None            # TODO: proporción de datos de x que son <= 13 (sin usar la ECDF)
mediana_ecdf = None    # TODO: primer valor de xs donde la ECDF llega a 0,5 o más
masa = completos(df, ["body_mass_g"])["body_mass_g"]
p_4000 = None          # TODO: proporción de pingüinos con masa <= 4000 g

# Comprobación automática
assert None not in (p_13, mediana_ecdf, p_4000), "Completa las tres líneas marcadas con TODO"
assert list(xs) == sorted(x) and np.isclose(ys[-1], 1.0), "xs debe estar ordenado y ys debe acabar en 1"
assert np.isclose(ys[0], 1 / 12), "ys debe empezar en 1/n"
assert np.isclose(p_13, 0.75), "ECDF(13) debe ser 9/12 = 0,75"
assert mediana_ecdf == 9, "la ECDF llega a 0,5 en el sexto dato ordenado (9)"
assert np.isclose(p_4000, np.searchsorted(np.sort(masa), 4000, side="right") / len(masa)), "revisa la proporción de masa <= 4000 g"
print("ECDF(13) =", p_13, "|", "Primer x con ECDF >= 0,5:", mediana_ecdf, "|", "Proporción con masa <= 4000 g:", round(p_4000, 3))
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Resumen por grupos

Completa `resumen_grupos(datos, col, grupo)`: devuelve un `DataFrame` con una fila por grupo y las columnas `n` (datos **no ausentes**), `mediana`, `Q1`, `Q3` e `IQR` (= Q3 − Q1). Se comprueba primero con un mini ejemplo con un ausente y luego con las especies de `penguins`.

In [ ]:
def resumen_grupos(datos, col, grupo):
    """Tabla por grupo: n (sin ausentes), mediana, Q1, Q3 e IQR de `col`."""
    # TODO: construye el DataFrame agrupando por `grupo`; usa count, median y quantile(0.25) / quantile(0.75)
    tabla = None
    # TODO: añade la columna IQR = Q3 - Q1
    return tabla

mini = pd.DataFrame({"g": ["a"] * 5 + ["b"] * 5, "v": [1, 2, 3, 4, 5, 10, 20, 30, 40, np.nan]})
r = resumen_grupos(mini, "v", "g")
assert r is not None, "Completa la función (TODO)"

# Comprobación automática
assert set(["n", "mediana", "Q1", "Q3", "IQR"]) <= set(r.columns), "faltan columnas: deben ser n, mediana, Q1, Q3, IQR"
assert (r.loc["a", "n"], r.loc["a", "mediana"], r.loc["a", "Q1"], r.loc["a", "Q3"]) == (5, 3, 2, 4), "con el mini ejemplo, el grupo a debe tener n=5, mediana=3, Q1=2, Q3=4"
assert (r.loc["b", "n"], r.loc["b", "mediana"], r.loc["b", "Q1"], r.loc["b", "Q3"]) == (4, 25, 17.5, 32.5), "el grupo b tiene 4 datos válidos (se excluye el ausente): mediana=25, Q1=17,5, Q3=32,5"

tabla_aleta = resumen_grupos(df, "flipper_length_mm", "species")
assert int(tabla_aleta["n"].sum()) == int(df["flipper_length_mm"].notna().sum()), "la suma de n debe ser el número de datos no ausentes"
assert np.allclose(tabla_aleta["IQR"], tabla_aleta["Q3"] - tabla_aleta["Q1"]), "IQR = Q3 - Q1"
display(tabla_aleta)

especie_max_iqr = None    # TODO: especie (índice de tabla_aleta) con mayor IQR
assert especie_max_iqr is not None, "Completa las líneas marcadas con TODO"
assert tabla_aleta.loc[especie_max_iqr, "IQR"] == tabla_aleta["IQR"].max(), "la especie de mayor IQR de aleta no coincide"
print("Especie con mayor IQR de longitud de aleta:", especie_max_iqr)
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Elegir el gráfico según la pregunta

Para cada pregunta, escribe el gráfico más adecuado: `"histograma"`, `"kde"`, `"boxplot"` o `"ecdf"`. Hay una respuesta claramente mejor en tres de ellas y dos válidas en una.

In [ ]:
# TODO: elige un gráfico para cada pregunta: 'histograma', 'kde', 'boxplot' o 'ecdf'
respuestas = {
    "¿Qué proporción de pingüinos pesa menos de 4000 g?": None,   # TODO
    "¿La longitud de aleta tiene uno o dos picos?": None,   # TODO
    "¿Cómo se comparan mediana y dispersión de la masa entre 6 grupos (especie × sexo)?": None,   # TODO
    "¿Cómo difieren las formas de la longitud de pico de las 3 especies, superpuestas?": None,   # TODO
}

validas = {
    "¿Qué proporción de pingüinos pesa menos de 4000 g?": {"ecdf"},
    "¿La longitud de aleta tiene uno o dos picos?": {"histograma", "kde"},
    "¿Cómo se comparan mediana y dispersión de la masa entre 6 grupos (especie × sexo)?": {"boxplot"},
    "¿Cómo difieren las formas de la longitud de pico de las 3 especies, superpuestas?": {"kde", "ecdf"},
}
assert None not in respuestas.values(), "Completa las cuatro líneas marcadas con TODO"
fallos = [p for p in validas if respuestas[p] not in validas[p]]
assert not fallos, "Revisa las respuestas de estas preguntas: " + " | ".join(fallos)
print("Ejercicio 5 correcto ✔")

---
## Reto final — Comparar tres especies sin engañar

Escribe la función `comparar_grupos(df, col)` que devuelva una **figura con dos paneles que comparten el eje x**: a la izquierda, una KDE por especie (`hue="species"`, `common_norm=False`); a la derecha, la ECDF por especie. Debe **excluir los ausentes de forma explícita** (con `completos`), poner un título con el *n* usado y etiquetar el eje x.

Después úsala con `bill_length_mm` y escribe tus conclusiones.

In [ ]:
def comparar_grupos(df, col):
    datos = None    # TODO: elimina los ausentes de `col` (usa completos)
    fig, ejes = None, None    # TODO: crea 1 fila y 2 columnas de ejes que compartan el eje x
    # TODO: KDE por especie (hue) SIN normalizar conjuntamente (common_norm=False) en ejes[0]
    # TODO: ECDF por especie (hue) en ejes[1]
    # TODO: pon un título a la figura con el n usado (fig.suptitle) y etiqueta los ejes
    return fig

assert comparar_grupos(df, "bill_length_mm") is not None, "Completa la función comparar_grupos (TODO)"
fig = comparar_grupos(df, "bill_length_mm")
plt.show()

assert len(fig.axes) == 2, "la figura debe tener 2 ejes"
assert fig.axes[0].get_shared_x_axes().joined(fig.axes[0], fig.axes[1]), "los dos ejes deben compartir el eje x"
assert all(len(ax.lines) >= 3 for ax in fig.axes), "cada eje debe dibujar 3 curvas (una por especie)"
assert fig._suptitle is not None and fig._suptitle.get_text().strip() != "", "pon un título a la figura (fig.suptitle)"
assert all(ax.get_xlabel().strip() != "" for ax in fig.axes), "etiqueta el eje x de los dos paneles"
display(resumen_grupos(df, "bill_length_mm", "species"))
print("Reto final correcto ✔")

**Tus conclusiones (3-4 frases):** ¿qué especie tiene las longitudes de pico más pequeñas? ¿Se solapan las distribuciones? ¿Qué gráfico te ha resultado más útil para responder y por qué? ¿Qué no puedes concluir solo con estos gráficos?

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
df, origen = cargar_penguins()
print("Origen de los datos:", origen)

faltantes = df.isna().sum()
n_filas_completas = int(df.notna().all(axis=1).sum())
col_mas_ausentes = faltantes.idxmax()
aleta = completos(df, ["flipper_length_mm"])["flipper_length_mm"]

assert int(faltantes.sum()) == int(df.isna().to_numpy().sum()), "faltantes debe contar los ausentes de cada columna"
assert n_filas_completas == int(df.notna().all(axis=1).sum()), "n_filas_completas: filas sin ningún ausente"
assert faltantes[col_mas_ausentes] == faltantes.max(), "col_mas_ausentes debe ser la columna con más ausentes"
assert aleta.notna().all() and len(aleta) == int(df["flipper_length_mm"].notna().sum()), "aleta debe ser la columna de aleta sin ausentes"
print("Columna con más ausentes:", col_mas_ausentes, "|", "Filas completas:", n_filas_completas)
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
x = np.array([2, 3, 7, 8, 8, 9, 12, 13, 13, 14, 21, 22])

bordes = np.arange(0, 30, 5)
conteos, _ = np.histogram(x, bins=bordes)
proporciones = conteos / len(x)
conteos10, _ = np.histogram(x, bins=np.arange(0, 40, 10))

assert list(conteos) == [2, 4, 4, 0, 2], "los conteos con ancho 5 deben ser [2, 4, 4, 0, 2]"
assert np.isclose(np.sum(proporciones), 1.0), "las proporciones deben sumar 1"
assert list(conteos10) == [6, 4, 2], "con ancho 10 los conteos deben ser [6, 4, 2]"
print("Con ancho 5 se ve el hueco [15,20); con ancho 10 desaparece.")
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
def ecdf(valores):
    """Devuelve (x ordenados, proporción acumulada) para cada dato."""
    xs = np.sort(np.asarray(valores))
    ys = np.arange(1, len(xs) + 1) / len(xs)
    return xs, ys

xs, ys = ecdf(x)

p_13 = float((x <= 13).mean())
mediana_ecdf = xs[np.argmax(ys >= 0.5)]
masa = completos(df, ["body_mass_g"])["body_mass_g"]
p_4000 = float((masa <= 4000).mean())

assert list(xs) == sorted(x) and np.isclose(ys[-1], 1.0), "xs debe estar ordenado y ys debe acabar en 1"
assert np.isclose(ys[0], 1 / 12), "ys debe empezar en 1/n"
assert np.isclose(p_13, 0.75), "ECDF(13) debe ser 9/12 = 0,75"
assert mediana_ecdf == 9, "la ECDF llega a 0,5 en el sexto dato ordenado (9)"
assert np.isclose(p_4000, np.searchsorted(np.sort(masa), 4000, side="right") / len(masa)), "revisa la proporción de masa <= 4000 g"
print("ECDF(13) =", p_13, "|", "Primer x con ECDF >= 0,5:", mediana_ecdf, "|", "Proporción con masa <= 4000 g:", round(p_4000, 3))
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
def resumen_grupos(datos, col, grupo):
    """Tabla por grupo: n (sin ausentes), mediana, Q1, Q3 e IQR de `col`."""
    g = datos.groupby(grupo)[col]
    tabla = pd.DataFrame({
        "n": g.count(),
        "mediana": g.median(),
        "Q1": g.quantile(0.25),
        "Q3": g.quantile(0.75),
    })
    tabla["IQR"] = tabla["Q3"] - tabla["Q1"]
    return tabla

mini = pd.DataFrame({"g": ["a"] * 5 + ["b"] * 5, "v": [1, 2, 3, 4, 5, 10, 20, 30, 40, np.nan]})
r = resumen_grupos(mini, "v", "g")

assert set(["n", "mediana", "Q1", "Q3", "IQR"]) <= set(r.columns), "faltan columnas: deben ser n, mediana, Q1, Q3, IQR"
assert (r.loc["a", "n"], r.loc["a", "mediana"], r.loc["a", "Q1"], r.loc["a", "Q3"]) == (5, 3, 2, 4), "con el mini ejemplo, el grupo a debe tener n=5, mediana=3, Q1=2, Q3=4"
assert (r.loc["b", "n"], r.loc["b", "mediana"], r.loc["b", "Q1"], r.loc["b", "Q3"]) == (4, 25, 17.5, 32.5), "el grupo b tiene 4 datos válidos (se excluye el ausente): mediana=25, Q1=17,5, Q3=32,5"

tabla_aleta = resumen_grupos(df, "flipper_length_mm", "species")
assert int(tabla_aleta["n"].sum()) == int(df["flipper_length_mm"].notna().sum()), "la suma de n debe ser el número de datos no ausentes"
assert np.allclose(tabla_aleta["IQR"], tabla_aleta["Q3"] - tabla_aleta["Q1"]), "IQR = Q3 - Q1"
display(tabla_aleta)

especie_max_iqr = tabla_aleta["IQR"].idxmax()
assert tabla_aleta.loc[especie_max_iqr, "IQR"] == tabla_aleta["IQR"].max(), "la especie de mayor IQR de aleta no coincide"
print("Especie con mayor IQR de longitud de aleta:", especie_max_iqr)
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
respuestas = {
    "¿Qué proporción de pingüinos pesa menos de 4000 g?": "ecdf",
    "¿La longitud de aleta tiene uno o dos picos?": "histograma",
    "¿Cómo se comparan mediana y dispersión de la masa entre 6 grupos (especie × sexo)?": "boxplot",
    "¿Cómo difieren las formas de la longitud de pico de las 3 especies, superpuestas?": "kde",
}

validas = {
    "¿Qué proporción de pingüinos pesa menos de 4000 g?": {"ecdf"},
    "¿La longitud de aleta tiene uno o dos picos?": {"histograma", "kde"},
    "¿Cómo se comparan mediana y dispersión de la masa entre 6 grupos (especie × sexo)?": {"boxplot"},
    "¿Cómo difieren las formas de la longitud de pico de las 3 especies, superpuestas?": {"kde", "ecdf"},
}
fallos = [p for p in validas if respuestas[p] not in validas[p]]
assert not fallos, "Revisa las respuestas de estas preguntas: " + " | ".join(fallos)
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
def comparar_grupos(df, col):
    datos = completos(df, [col, "species"])
    fig, ejes = plt.subplots(1, 2, figsize=(12, 3.8), sharex=True)
    sns.kdeplot(data=datos, x=col, hue="species", common_norm=False, ax=ejes[0])
    sns.ecdfplot(data=datos, x=col, hue="species", ax=ejes[1])
    fig.suptitle(f"Longitud de pico por especie: n = {len(datos)}")
    for ax in ejes:
        ax.set_xlabel("Longitud de pico (mm)")
    return fig

fig = comparar_grupos(df, "bill_length_mm")
plt.show()

assert len(fig.axes) == 2, "la figura debe tener 2 ejes"
assert fig.axes[0].get_shared_x_axes().joined(fig.axes[0], fig.axes[1]), "los dos ejes deben compartir el eje x"
assert all(len(ax.lines) >= 3 for ax in fig.axes), "cada eje debe dibujar 3 curvas (una por especie)"
assert fig._suptitle is not None and fig._suptitle.get_text().strip() != "", "pon un título a la figura (fig.suptitle)"
assert all(ax.get_xlabel().strip() != "" for ax in fig.axes), "etiqueta el eje x de los dos paneles"
display(resumen_grupos(df, "bill_length_mm", "species"))
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*